<a href="https://colab.research.google.com/github/mikakia/FL_VIP/blob/main/knee_images.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [ ]:
import os
from collections import Counter

import matplotlib.pyplot as plt
from scipy.stats import chisquare, f_oneway, ks_2samp
from torchvision import datasets, transforms


# Load data

In [ ]:
folder_dir = '/Users/tsampikakiaourtzi/Google Drive/My Drive/FL-VIP/knee_images'

In [ ]:
print("Exists:", os.path.exists(folder_dir))
print("folder_dir =", folder_dir)

In [ ]:
knee_images = datasets.ImageFolder(root=folder_dir, transform=transforms.ToTensor())
print(type(knee_images[0]))

# Explore Data

In [ ]:
print(f"Classes:{knee_images.classes}")
print(f"Class and Index: {knee_images.class_to_idx}")
print(f"Total images: {len(knee_images)}")

In [ ]:
counts = Counter(knee_images.targets)
for class_idx, count in sorted(counts.items()):
    print(f"Class {knee_images.classes[class_idx]}: {count} images")

In [ ]:
image, label = knee_images[0]
print(image.shape)
print(label)

## Plot of count of images per class

In [ ]:
class_names = knee_images.classes
values = [counts[i] for i in range(len(class_names))]


plt.figure(figsize=(8, 5))
bars = plt.bar(class_names, values, color='purple', edgecolor='black')

for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
             str(val), ha='center', va='bottom', fontsize=11)

plt.title('Number of Images per Class')
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.tight_layout()
plt.show()

## Plot of an example image from each class

In [ ]:


class_labels = {
    0: 'KL 0 (Healthy)',
    1: 'KL 1 (Mostly Healthy)',
    2: 'KL 2 (Doubtful OA)',
    3: 'KL 3 (Knee OA)',
    4: 'KL 4 (Severe OA)'
}

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

count = 0
for class_idx in range(len(knee_images.classes)):
    count = count + 1
    print("count", count)

    idx = knee_images.targets.index(class_idx)
    print("index", idx)

    img, label = knee_images[idx]

    axes[class_idx].imshow(img.permute(1, 2, 0))
    axes[class_idx].set_title(f'{class_labels[class_idx]}')
    axes[class_idx].axis('off')

plt.suptitle('Example from each class\n', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Class Distribution

In [ ]:
label_counts = Counter(knee_images.targets)
labels_sorted = sorted(label_counts.items())

plt.figure(figsize=(8, 5))
plt.bar([class_labels[k] for k, v in labels_sorted], [v for k, v in labels_sorted],color='purple',edgecolor='blue')
plt.xticks(rotation=30, ha='right')
plt.ylabel('Number of images')
plt.title('Class Distribution')
plt.tight_layout()
plt.show()

for k, v in labels_sorted:
    print(f"{class_labels[k]}: {v} images ({v/len(knee_images.targets)*100:.1f}%)")

# Pie chart of class distribution

In [ ]:
sizes = [v for k, v in labels_sorted]
labels = [class_labels[k] for k, v in labels_sorted]
colors = plt.cm.YlOrRd([i / (len(sizes)-1) for i in range(len(sizes))])

plt.figure(figsize=(8, 8))
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90, colors=colors)
plt.title('Class Distribution (KL Grades)\n\n', fontsize=14, fontweight='bold')
plt.axis('equal')
plt.show()

# Chi-squared test
For class imbalance
This test compares the actual number of images in each KL grade class (observed values) against the number we would expect if all five classes were equally represented (expected values under a uniform distribution).

Result: The chi-square test gave a statistic of 2729.56 and a p-value of less than 0.0001, showing that the number of images in each class is very different from what we'd expect if the classes were equal in size. Because this p-value is so small, we can say with confidence that this imbalance is real and not just by chance. 

Statistic: how far your actual class counts are from what you'd expect if all 5 classes had equal numbers of images. 
χ² = Σ [ (observed - expected)² / expected ]


In [ ]:
observed = [v for k, v in labels_sorted]
expected = [sum(observed)/len(observed)] * len(observed)

stat, p = chisquare(observed, expected)
print(f"Chi-square: {stat:.2f}, p-value: {p:.4e}")

# ANOVA across all the classes and pairwise between the classes (for mean brightness)
Mean pixel intensit, not on overall image brightness. Two X-rays can have identical average pixel intensity while looking completely different in terms of joint structure.
Comparison	KS stat	p-value	Significant
Mean pixel intensity alone is not a good discriminator between KL grades. This makes total sense clinically too: KL grading is based on structural features (joint space narrowing, osteophytes, bone changes) — not on overall image brightness. Two X-rays can have identical average pixel intensity while looking completely different in terms of joint structure.
Comparison	KS stat	p-value	Significant?
KL0 vs KL1	0.180	0.3959	No
KL1 vs KL2	0.140	0.7166	No
KL2 vs KL3	0.160	0.5487	No
KL3 vs KL4	0.180	0.3959	No


In [ ]:


def get_mean_intensities(class_idx, n_samples=50):
    indices = [i for i, t in enumerate(knee_images.targets) if t == class_idx][:n_samples]
    return [knee_images[i][0].numpy().mean() for i in indices]

class_intensities = {c: get_mean_intensities(c) for c in range(5)} # mean brightness

# ANOVA across all classes
stat, p = f_oneway(*class_intensities.values())
print(f"ANOVA: F={stat:.2f}, p={p:.4e}")

# Pairwise KS test between classes
for a, b in [(0,1), (1,2), (2,3), (3,4)]:
    stat, p = ks_2samp(class_intensities[a], class_intensities[b])
    print(f"KL{a} vs KL{b}: KS stat={stat:.3f}, p={p:.4f}")


# ANOVA on Contrast (Pixel Standard Deviation)

The F-statistic, F = 1.04, and the p-value, p = 0.39, was much higher than 0.05, meaning the contrast levels between the different KL grades are not meaningfully different, and any  differences observed are likely just due to random variation and not a real pattern.
No significant difference in contrast between the classes either. 


In [ ]:
def get_std_intensities(class_idx, n_samples=30):
    indices = [i for i, t in enumerate(knee_images.targets) if t == class_idx][:n_samples]
    return [knee_images[i][0].numpy().std() for i in indices]

class_stds = {c: get_std_intensities(c) for c in range(5)} # contrast values from each class
stat, p = f_oneway(*class_stds.values())
print(f"ANOVA on contrast: F={stat:.2f}, p={p:.4e}")

Overall: The differences between KL grades are not about overall lighting/contrast, but about specific structural details located in particular regions of the image..

More to come